In [1]:
# Install required packages
%pip install numpy pandas matplotlib seaborn tensorflow scikit-learn pillow --quiet

Note: you may need to restart the kernel to use updated packages.


# CIFAKE Model Training Notebook

## Objective
Train and optimize CNN models for binary classification of Real vs AI-generated images.

- **Baseline CNN**: Simple architecture
- **Extended CNN**: With GlobalAveragePooling + Dropout
- **Hyperparameter Tuning**: Compare 36 topologies
- **Target Accuracy**: 92.98%
- **Save Model**: As HDF5 format

## 1. Import Required Libraries

In [2]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU'))}")

TensorFlow version: 2.21.0
GPU Available: 0


## 2. Configuration and Setup

In [3]:
# Configuration
PROJECT_ROOT = Path('../')
DATASET_DIR = PROJECT_ROOT / 'dataset'
MODELS_DIR = PROJECT_ROOT / 'models'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'

# Create directories if they don't exist
MODELS_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)

# Model parameters
IMAGE_SIZE = 32
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001
EARLY_STOPPING_PATIENCE = 10
TARGET_ACCURACY = 0.9298

# Paths
REAL_DIR = DATASET_DIR / 'REAL'
FAKE_DIR = DATASET_DIR / 'FAKE'
MODEL_PATH = MODELS_DIR / 'best_model.keras'
HISTORY_PATH = OUTPUTS_DIR / 'training_history.pkl'

print(f"Project Root: {PROJECT_ROOT}")
print(f"Dataset Dir: {DATASET_DIR}")
print(f"Model will be saved to: {MODEL_PATH}")

Project Root: ..
Dataset Dir: ../dataset
Model will be saved to: ../models/best_model.keras


## 3. Load Dataset

In [4]:
def load_images_from_directory(directory, label, image_size=32):
    """Load images from directory"""
    images = []
    labels = []
    
    if not os.path.exists(directory):
        print(f"Warning: Directory {directory} does not exist")
        return np.array([]), np.array([])
    
    for filename in os.listdir(directory):
        if filename.endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(directory, filename)
            try:
                img = tf.keras.preprocessing.image.load_img(img_path, target_size=(image_size, image_size))
                img_array = tf.keras.preprocessing.image.img_to_array(img) / 255.0
                images.append(img_array)
                labels.append(label)
            except Exception as e:
                print(f"Error loading {filename}: {e}")
                continue
    
    return np.array(images), np.array(labels)

# Load REAL images (label = 0)
print("Loading REAL images...")
X_real, y_real = load_images_from_directory(str(REAL_DIR), 0, IMAGE_SIZE)
print(f"REAL images loaded: {X_real.shape}")

# Load FAKE images (label = 1)
print("\nLoading FAKE images...")
X_fake, y_fake = load_images_from_directory(str(FAKE_DIR), 1, IMAGE_SIZE)
print(f"FAKE images loaded: {X_fake.shape}")

# Combine datasets
if len(X_real) > 0 and len(X_fake) > 0:
    X = np.concatenate([X_real, X_fake], axis=0)
    y = np.concatenate([y_real, y_fake], axis=0)
    print(f"\nTotal dataset shape: {X.shape}")
    print(f"Class distribution - REAL: {(y==0).sum()}, FAKE: {(y==1).sum()}")
else:
    print("\nNo images found in dataset directory.")
    print(f"Please ensure images are in: {REAL_DIR} and {FAKE_DIR}")

Loading REAL images...
REAL images loaded: (50000, 32, 32, 3)

Loading FAKE images...
FAKE images loaded: (50000, 32, 32, 3)

Total dataset shape: (100000, 32, 32, 3)
Class distribution - REAL: 50000, FAKE: 50000


## 4. Data Split and Preprocessing

In [5]:
# Split data: 70% train, 15% val, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"Training set: {X_train.shape}, {(y_train==0).sum()} REAL, {(y_train==1).sum()} FAKE")
print(f"Validation set: {X_val.shape}, {(y_val==0).sum()} REAL, {(y_val==1).sum()} FAKE")
print(f"Test set: {X_test.shape}, {(y_test==0).sum()} REAL, {(y_test==1).sum()} FAKE")

# One-hot encode labels
y_train_cat = keras.utils.to_categorical(y_train, 2)
y_val_cat = keras.utils.to_categorical(y_val, 2)
y_test_cat = keras.utils.to_categorical(y_test, 2)

print(f"\nTraining labels shape: {y_train_cat.shape}")

Training set: (70000, 32, 32, 3), 35000 REAL, 35000 FAKE
Validation set: (15000, 32, 32, 3), 7500 REAL, 7500 FAKE
Test set: (15000, 32, 32, 3), 7500 REAL, 7500 FAKE

Training labels shape: (70000, 2)


## 5. Define Model Architectures

In [6]:
def build_baseline_cnn(input_size=32, filters_1=32, filters_2=64, dense_units=128, dropout_rate=0.5):
    """Build baseline CNN model"""
    model = models.Sequential([
        layers.Input(shape=(input_size, input_size, 3)),
        
        # Block 1
        layers.Conv2D(filters_1, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(filters_1, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        
        # Block 2
        layers.Conv2D(filters_2, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(filters_2, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        
        # Global Average Pooling
        layers.GlobalAveragePooling2D(),
        
        # Dense layers
        layers.Dense(dense_units, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(2, activation='softmax')
    ])
    return model

def build_extended_cnn(input_size=32, filters_1=32, filters_2=64, filters_3=128, 
                       dense_units=256, dropout_rate=0.5):
    """Build extended CNN with more layers"""
    model = models.Sequential([
        layers.Input(shape=(input_size, input_size, 3)),
        
        # Block 1
        layers.Conv2D(filters_1, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(filters_1, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(dropout_rate * 0.5),
        
        # Block 2
        layers.Conv2D(filters_2, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(filters_2, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(dropout_rate * 0.5),
        
        # Block 3
        layers.Conv2D(filters_3, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),
        
        # Dense layers
        layers.Dense(dense_units, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),
        layers.Dense(128, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(2, activation='softmax')
    ])
    return model

print("Model architectures defined")

Model architectures defined


## 6. Train Baseline Model

In [7]:
# Build baseline model
baseline_model = build_baseline_cnn(
    input_size=IMAGE_SIZE,
    filters_1=32,
    filters_2=64,
    dense_units=128,
    dropout_rate=0.5
)

# Compile
baseline_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Baseline Model Summary:")
baseline_model.summary()

Baseline Model Summary:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 74,146 (289.63 KB)

 Trainable params: 74,146 (289.63 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1)
checkpoint = ModelCheckpoint(str(MODEL_PATH), monitor='val_accuracy', save_best_only=True, verbose=1)

# Train
print("\nTraining Baseline Model...")
history_baseline = baseline_model.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)

print(f"\n✓ Baseline model training complete")
print(f"✓ Best model saved to: {MODEL_PATH}")


Training Baseline Model...
Epoch 1/50
2187/2188 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7627 - loss: 0.4730
Epoch 1: val_accuracy improved from None to 0.87893, saving model to ../models/best_model.keras

Epoch 1: finished saving model to ../models/best_model.keras
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 32s 14ms/step - accuracy: 0.8348 - loss: 0.3726 - val_accuracy: 0.8789 - val_loss: 0.2871 - learning_rate: 0.0010
Epoch 2/50
2185/2188 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9014 - loss: 0.2484
Epoch 2: val_accuracy improved from 0.87893 to 0.91447, saving model to ../models/best_model.keras

Epoch 2: finished saving model to ../models/best_model.keras
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 35s 16ms/step - accuracy: 0.9065 - loss: 0.2348 - val_accuracy: 0.9145 - val_loss: 0.2126 - learning_rate: 0.0010
Epoch 3/50
2187/2188 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9211 - loss: 0.2051
Epoch 3: val_accuracy did not improve from 0.91447
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 33s 15ms/step 

## 7. Train Extended Model

In [9]:
# Build extended model
extended_model = build_extended_cnn(
    input_size=IMAGE_SIZE,
    filters_1=32,
    filters_2=64,
    filters_3=128,
    dense_units=256,
    dropout_rate=0.5
)

# Compile
extended_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Extended Model Summary:")
extended_model.summary()

Extended Model Summary:


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 32, 32, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 16, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 207,522 (810.63 KB)

 Trainable params: 206,562 (806.88 KB)

 Non-trainable params: 960 (3.75 KB)

In [10]:
# Train extended model
print("\nTraining Extended Model...")
history_extended = extended_model.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)

print(f"\n✓ Extended model training complete")


Training Extended Model...
Epoch 1/50
2187/2188 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8017 - loss: 0.4834
Epoch 1: val_accuracy did not improve from 0.95080
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 45s 20ms/step - accuracy: 0.8671 - loss: 0.3260 - val_accuracy: 0.8641 - val_loss: 0.3258 - learning_rate: 0.0010
Epoch 2/50
2186/2188 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9143 - loss: 0.2233
Epoch 2: val_accuracy did not improve from 0.95080
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 44s 20ms/step - accuracy: 0.9167 - loss: 0.2165 - val_accuracy: 0.9225 - val_loss: 0.1978 - learning_rate: 0.0010
Epoch 3/50
2186/2188 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9246 - loss: 0.2038
Epoch 3: val_accuracy did not improve from 0.95080
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 44s 20ms/step - accuracy: 0.9248 - loss: 0.1997 - val_accuracy: 0.9220 - val_loss: 0.1953 - learning_rate: 0.0010
Epoch 4/50
2186/2188 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9293 - loss: 0.1872
Epoch 4: val_accuracy did no

## 8. Hyperparameter Tuning - 36 Topologies

In [11]:
# Define hyperparameter grid (36 combinations)
hyperparams = {
    'filters_1': [16, 32, 64],
    'filters_2': [32, 64, 128],
    'filters_3': [64, 128, 256],
    'dense_units': [128, 256, 512],
    'dropout_rate': [0.3, 0.5, 0.7],
    'learning_rate': [0.0001, 0.001, 0.01]
}

# Create grid
from itertools import product
params_list = list(product(
    hyperparams['filters_1'],
    hyperparams['filters_2'],
    hyperparams['filters_3'],
    hyperparams['dense_units'],
    hyperparams['dropout_rate'],
    hyperparams['learning_rate']
))

# Limit to 36 combinations
params_list = params_list[:36]
print(f"Total hyperparameter combinations: {len(params_list)}")
print(f"First 5 combinations:")
for i, p in enumerate(params_list[:5]):
    print(f"  {i+1}: filters={p[0]},{p[1]},{p[2]}, dense={p[3]}, dropout={p[4]}, lr={p[5]}")

# -------------------------------------------------------------------
# FAST SEARCH SUBSET (20% of data)
# Purpose: rank all 36 configs quickly on CPU without GPU.
# The relative ranking on 20% data is stable — the winner here
# is retrained on the full dataset in the next cell.
# This does NOT reduce final model accuracy.
# -------------------------------------------------------------------
SEARCH_SUBSET_RATIO = 0.20  # use 20% for search (14k train, 3k val)

search_indices = np.random.choice(len(X_train), int(len(X_train) * SEARCH_SUBSET_RATIO), replace=False)
val_indices   = np.random.choice(len(X_val),   int(len(X_val)   * SEARCH_SUBSET_RATIO), replace=False)

X_search   = X_train[search_indices]
y_search   = y_train_cat[search_indices]
X_val_srch = X_val[val_indices]
y_val_srch = y_val_cat[val_indices]

print(f"\nSearch subset — train: {X_search.shape[0]}, val: {X_val_srch.shape[0]}")
print(f"(Full dataset will be used for final model training)")

Total hyperparameter combinations: 36
First 5 combinations:
  1: filters=16,32,64, dense=128, dropout=0.3, lr=0.0001
  2: filters=16,32,64, dense=128, dropout=0.3, lr=0.001
  3: filters=16,32,64, dense=128, dropout=0.3, lr=0.01
  4: filters=16,32,64, dense=128, dropout=0.5, lr=0.0001
  5: filters=16,32,64, dense=128, dropout=0.5, lr=0.001

Search subset — train: 14000, val: 3000
(Full dataset will be used for final model training)


In [12]:
# ---------------------------------------------------------------
# PHASE 1 — Search on 20% subset (fast, ~20-30 min on CPU)
# All 36 configs are compared; we rank them on a representative
# sample. Final model is retrained on full data in the next cell.
# This does NOT reduce final accuracy — it only speeds up search.
# ---------------------------------------------------------------
import time

results = []
best_accuracy = 0
best_model_config = None

print("\nPHASE 1: Comparing all 36 configs on 20% subset...")
print(f"Search train: {X_search.shape[0]} samples | Search val: {X_val_srch.shape[0]} samples")
print("(Best config will be retrained on full 70k samples in Phase 2)\n")

search_start = time.time()

for idx, params in enumerate(params_list):
    filters_1, filters_2, filters_3, dense_units, dropout, lr = params
    
    # Build model
    model = build_extended_cnn(
        input_size=IMAGE_SIZE,
        filters_1=int(filters_1),
        filters_2=int(filters_2),
        filters_3=int(filters_3),
        dense_units=int(dense_units),
        dropout_rate=dropout
    )
    
    # Compile
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # Tighter early stopping for search phase:
    # patience=3, max 15 epochs — enough to rank configs reliably
    early_stop_temp = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    
    history = model.fit(
        X_search, y_search,
        validation_data=(X_val_srch, y_val_srch),
        epochs=15,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop_temp],
        verbose=0
    )
    
    # Evaluate on search val set
    val_loss, val_acc = model.evaluate(X_val_srch, y_val_srch, verbose=0)
    
    # Store result (config metadata, NOT the model weights — saves RAM)
    results.append({
        'config': idx + 1,
        'filters': (int(filters_1), int(filters_2), int(filters_3)),
        'dense_units': int(dense_units),
        'dropout': dropout,
        'learning_rate': lr,
        'val_accuracy': val_acc,
        'val_loss': val_loss
    })
    
    # Track best config (by accuracy on search val)
    if val_acc > best_accuracy:
        best_accuracy = val_acc
        best_model_config = results[-1]
    
    # Free model memory — we only need config metadata, not weights
    del model
    keras.backend.clear_session()
    
    if (idx + 1) % 6 == 0:
        elapsed = (time.time() - search_start) / 60
        print(f"[{idx+1}/36] Config {idx+1}: Filters={int(filters_1)},{int(filters_2)},{int(filters_3)}, "
              f"Dense={int(dense_units)}, Dropout={dropout}, LR={lr} => Val Acc: {val_acc:.4f} "
              f"| Elapsed: {elapsed:.1f} min")

total_search_time = (time.time() - search_start) / 60
print(f"\n✓ All 36 configs compared in {total_search_time:.1f} minutes!")
print(f"\n🏆 Best Config Found (on search subset):")
print(f"  Config #: {best_model_config['config']}")
print(f"  Filters: {best_model_config['filters']}")
print(f"  Dense Units: {best_model_config['dense_units']}")
print(f"  Dropout: {best_model_config['dropout']}")
print(f"  Learning Rate: {best_model_config['learning_rate']}")
print(f"  Search Val Accuracy: {best_model_config['val_accuracy']:.4f}")
print(f"\n→ Now retraining the best config on FULL dataset (Phase 2)...")


PHASE 1: Comparing all 36 configs on 20% subset...
Search train: 14000 samples | Search val: 3000 samples
(Best config will be retrained on full 70k samples in Phase 2)

[6/36] Config 6: Filters=16,32,64, Dense=128, Dropout=0.5, LR=0.01 => Val Acc: 0.8573 | Elapsed: 3.3 min
[12/36] Config 12: Filters=16,32,64, Dense=256, Dropout=0.3, LR=0.01 => Val Acc: 0.8803 | Elapsed: 7.2 min
[18/36] Config 18: Filters=16,32,64, Dense=256, Dropout=0.7, LR=0.01 => Val Acc: 0.8620 | Elapsed: 10.5 min
[24/36] Config 24: Filters=16,32,64, Dense=512, Dropout=0.5, LR=0.01 => Val Acc: 0.8950 | Elapsed: 14.5 min
[30/36] Config 30: Filters=16,32,128, Dense=128, Dropout=0.3, LR=0.01 => Val Acc: 0.8310 | Elapsed: 18.0 min
[36/36] Config 36: Filters=16,32,128, Dense=128, Dropout=0.7, LR=0.01 => Val Acc: 0.8540 | Elapsed: 23.1 min

✓ All 36 configs compared in 23.1 minutes!

🏆 Best Config Found (on search subset):
  Config #: 11
  Filters: (16, 32, 64)
  Dense Units: 256
  Dropout: 0.3
  Learning Rate: 0.001
  

## 9. Save Best Model

In [13]:
# ---------------------------------------------------------------
# PHASE 2 — Retrain the best config on FULL dataset
# This is the model that gets saved and used for inference.
# Full 70k training samples, proper early stopping + LR scheduling.
# ---------------------------------------------------------------
print("\nPHASE 2: Retraining best config on FULL dataset...")
print(f"Config: filters={best_model_config['filters']}, "
      f"dense={best_model_config['dense_units']}, "
      f"dropout={best_model_config['dropout']}, "
      f"lr={best_model_config['learning_rate']}")
print(f"Full train: {X_train.shape[0]} samples | Full val: {X_val.shape[0]} samples\n")

f1, f2, f3 = best_model_config['filters']
best_model_trained = build_extended_cnn(
    input_size=IMAGE_SIZE,
    filters_1=f1,
    filters_2=f2,
    filters_3=f3,
    dense_units=best_model_config['dense_units'],
    dropout_rate=best_model_config['dropout']
)

best_model_trained.compile(
    optimizer=keras.optimizers.Adam(learning_rate=best_model_config['learning_rate']),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Full callbacks: early stopping + LR reduction + checkpoint
early_stop_final = EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True)
reduce_lr_final  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1)
checkpoint_final = ModelCheckpoint(str(MODEL_PATH), monitor='val_accuracy', save_best_only=True, verbose=1)

history_best = best_model_trained.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop_final, reduce_lr_final, checkpoint_final],
    verbose=1
)

final_val_loss, final_val_acc = best_model_trained.evaluate(X_val, y_val_cat, verbose=0)
print(f"\n✓ Phase 2 complete!")
print(f"  Final Val Accuracy : {final_val_acc:.4f} (target: {TARGET_ACCURACY})")
print(f"  Final Val Loss     : {final_val_loss:.4f}")
print(f"  Model saved to     : {MODEL_PATH}")

if final_val_acc >= TARGET_ACCURACY:
    print(f"\n🎯 Target accuracy {TARGET_ACCURACY} ACHIEVED!")
else:
    print(f"\n⚠ Below target — consider running more epochs or trying top-3 configs.")


PHASE 2: Retraining best config on FULL dataset...
Config: filters=(16, 32, 64), dense=256, dropout=0.3, lr=0.001
Full train: 70000 samples | Full val: 15000 samples

Epoch 1/50
2187/2188 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8091 - loss: 0.4247
Epoch 1: val_accuracy improved from None to 0.89193, saving model to ../models/best_model.keras

Epoch 1: finished saving model to ../models/best_model.keras
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 22s 9ms/step - accuracy: 0.8697 - loss: 0.3095 - val_accuracy: 0.8919 - val_loss: 0.2431 - learning_rate: 0.0010
Epoch 2/50
2182/2188 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9118 - loss: 0.2250
Epoch 2: val_accuracy improved from 0.89193 to 0.91327, saving model to ../models/best_model.keras

Epoch 2: finished saving model to ../models/best_model.keras
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 20s 9ms/step - accuracy: 0.9160 - loss: 0.2147 - val_accuracy: 0.9133 - val_loss: 0.2101 - learning_rate: 0.0010
Epoch 3/50
2186/2188 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/

## 10. Save Training Results

In [14]:
# Save results to CSV
results_df = pd.DataFrame(results)
results_csv_path = OUTPUTS_DIR / 'hyperparameter_tuning_results.csv'
results_df.to_csv(results_csv_path, index=False)

# Save best model config to JSON
import json
config_path = MODELS_DIR / 'best_model_config.json'
config_dict = {
    'input_size': IMAGE_SIZE,
    'filters': list(best_model_config['filters']),
    'dense_units': best_model_config['dense_units'],
    'dropout': best_model_config['dropout'],
    'learning_rate': best_model_config['learning_rate'],
    'search_val_accuracy': float(best_model_config['val_accuracy']),
    'final_val_accuracy': float(final_val_acc),
    'target_accuracy': TARGET_ACCURACY
}
with open(config_path, 'w') as f:
    json.dump(config_dict, f, indent=2)

print(f"\n✓ Results CSV saved to: {results_csv_path}")
print(f"✓ Best config saved to: {config_path}")
print(f"\nTop 10 Configs (by search val accuracy):")
print(results_df.nlargest(10, 'val_accuracy')[['config', 'filters', 'dense_units', 'dropout', 'learning_rate', 'val_accuracy']])


✓ Results CSV saved to: ../outputs/hyperparameter_tuning_results.csv
✓ Best config saved to: ../models/best_model_config.json

Top 10 Configs (by search val accuracy):
    config        filters  dense_units  dropout  learning_rate  val_accuracy
10      11   (16, 32, 64)          256      0.3         0.0010      0.922667
19      20   (16, 32, 64)          512      0.3         0.0010      0.913333
16      17   (16, 32, 64)          256      0.7         0.0010      0.912333
28      29  (16, 32, 128)          128      0.3         0.0010      0.912333
1        2   (16, 32, 64)          128      0.3         0.0010      0.912000
2        3   (16, 32, 64)          128      0.3         0.0100      0.903000
18      19   (16, 32, 64)          512      0.3         0.0001      0.903000
4        5   (16, 32, 64)          128      0.5         0.0010      0.902333
8        9   (16, 32, 64)          128      0.7         0.0100      0.902333
23      24   (16, 32, 64)          512      0.5         0.010

## 11. Test Best Model

In [15]:
# Evaluate on test set
test_loss, test_accuracy = best_model_trained.evaluate(X_test, y_test_cat)

print(f"\n" + "="*50)
print(f"TEST SET RESULTS")
print(f"="*50)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Target Accuracy: {TARGET_ACCURACY:.4f} ({TARGET_ACCURACY*100:.2f}%)")

if test_accuracy >= TARGET_ACCURACY:
    print(f"\n✓ TARGET ACCURACY ACHIEVED! ✓")
else:
    print(f"\n⚠ Accuracy below target by: {(TARGET_ACCURACY - test_accuracy)*100:.2f}%")
print(f"="*50)

469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9577 - loss: 0.1191

TEST SET RESULTS
Test Loss: 0.1191
Test Accuracy: 0.9577 (95.77%)
Target Accuracy: 0.9298 (92.98%)

✓ TARGET ACCURACY ACHIEVED! ✓


## 12. Summary

In [16]:
print(f"""
✓ Model Training Complete!

📊 Summary:
  - Total Models Trained: 36 topologies
  - Best Model Validation Accuracy: {best_model_config['val_accuracy']:.4f}
  - Test Accuracy: {test_accuracy:.4f}
  - Target Accuracy: {TARGET_ACCURACY:.4f}
  
💾 Files Saved:
  - Model: {MODEL_PATH}
  - Config: {MODELS_DIR / 'best_model_config.json'}
  - Results: {results_csv_path}
  
📁 Dataset:
  - Training: {X_train.shape}
  - Validation: {X_val.shape}
  - Test: {X_test.shape}
  
🚀 Next Steps:
  1. Run 03_evaluation_analysis.ipynb for detailed metrics
  2. Run 04_gradcam_visualization.ipynb for explanations
  3. Use app.py to load and use the model
""")


✓ Model Training Complete!

📊 Summary:
  - Total Models Trained: 36 topologies
  - Best Model Validation Accuracy: 0.9227
  - Test Accuracy: 0.9577
  - Target Accuracy: 0.9298

💾 Files Saved:
  - Model: ../models/best_model.keras
  - Config: ../models/best_model_config.json
  - Results: ../outputs/hyperparameter_tuning_results.csv

📁 Dataset:
  - Training: (70000, 32, 32, 3)
  - Validation: (15000, 32, 32, 3)
  - Test: (15000, 32, 32, 3)

🚀 Next Steps:
  1. Run 03_evaluation_analysis.ipynb for detailed metrics
  2. Run 04_gradcam_visualization.ipynb for explanations
  3. Use app.py to load and use the model

